In [0]:
base = "/Volumes/dbw_dp750_anshul/pharma/raw"
display(dbutils.fs.ls(f"{base}/kanto_pharma/"))

In [0]:
df_jan = (spark.read
            .option("header", True)
            .csv(f"{base}/kanto_pharma/2026-01.csv"))
print(df_jan.count())
df_jan.printSchema()
display(df_jan)

In [0]:
from pyspark.sql import functions as F

df_kanto = (spark.read
              .option("header", True)
              .csv(f"{base}/kanto_pharma/*.csv")
              .select("*", F.col("_metadata.file_path").alias("source_file")))
print(df_kanto.count())
display(df_kanto.groupBy("source_file").count().orderBy("source_file"))

In [0]:
total = df_jan.count()
missing_codes = df_jan.filter(F.col("PRODUCT_CD").isNull()).count()
duplicates = total - df_jan.dropDuplicates().count()

print(f"Rows: {total}")
print(f"Missing product codes: {missing_codes}")
print(f"Duplicate rows: {duplicates}")

In [0]:
from pyspark.sql import functions as F

base = "/Volumes/dbw_dp750_anshul/pharma/raw"
distributors = ["alpha_medical", "kanto_pharma", "sakura_medlink"]

for dist in distributors:
    df = (spark.read
            .option("header", True)
            .csv(f"{base}/{dist}/*.csv"))

    # technical column-name cleanup only (Delta doesn't allow spaces/brackets)
    clean_cols = [c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
                  for c in df.columns]
    df = df.toDF(*clean_cols)

    # audit columns: where did each row come from, and when did we load it?
    df = (df.withColumn("source_file", F.col("_metadata.file_path"))
            .withColumn("ingested_at", F.current_timestamp()))

    table_name = f"dbw_dp750_anshul.pharma.bronze_{dist}"
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f"{table_name}: {spark.table(table_name).count()} rows")